In [1]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
from pathlib import Path
import seaborn as sns

#Read in input data
script_loc = Path('01_load_and_preprocess.py').resolve().parent
data_loc = script_loc.parent / "data"
resources_loc = script_loc.parent / 'resources'

In [2]:
#Read in preprocessed data
cdata = sc.read_h5ad(data_loc / 'intermediate/c_preprocessed.h5ad')
pdata = sc.read_h5ad(data_loc / 'intermediate/p_preprocessed.h5ad')

In [3]:
del cdata.raw
del pdata.raw

In [4]:
#Format for mapmycells
#By dropping all data except for the central sparse matrix and the var and obs assignments
map_cdata = cdata.copy()
map_cdata.obs = map_cdata.obs[[]]
map_cdata.var = map_cdata.var[[]]
del map_cdata.uns
map_cdata.layers.clear(keep_x=True)
map_cdata

AnnData object with n_obs × n_vars = 57571 × 31393
    layers: None (.X)

In [5]:
map_pdata = pdata.copy()
map_pdata.obs = map_pdata.obs[[]]
map_pdata.var = map_pdata.var[[]]
del map_pdata.uns
map_pdata.layers.clear(keep_x=True)
map_pdata

AnnData object with n_obs × n_vars = 41275 × 31393
    layers: None (.X)

In [6]:
#Write mapping formated objects to data/intermediate
map_pdata.write_h5ad(data_loc / 'intermediate/map_pdata.h5ad')


In [7]:
map_cdata.write_h5ad(data_loc / 'intermediate/map_cdata.h5ad')

In [8]:
#Run MapMyCells through https://knowledge.brain-map.org/mapmycells/process 
#Using Hierarchical Mapping to 10x Whole Human Brain reference taxonomy for both files

In [9]:
#Read in pdata cell assignments, skipping metadata rows and removing machine readable cluster assignments
pdata_cells = (pd.read_csv(resources_loc / 'map_pdata_10xWholeHumanBrain(CCN202210140)_HierarchicalMapping_UTC_1793473199081.csv', 
                           skiprows=4))
pdata_cells = (pdata_cells.drop(columns=['supercluster_label', 'cluster_label', 'subcluster_label', 'subcluster_alias'])
                            .set_index('cell_id'))

#Convert categorical data to be categorical
pdata_cells = pdata_cells.astype({'supercluster_name': 'category', 'cluster_name': 'category', 'subcluster_name': 'category'})
pdata_cells

,supercluster_name,supercluster_bootstrapping_probability,cluster_name,cluster_bootstrapping_probability,subcluster_name,subcluster_bootstrapping_probability
cell_id,,,,,,
AAACCCAAGGATACCG_11,Microglia,1.0,Mgl_11,0.59,Mgl_11_3285,0.38
AAACCCAAGGGACCAT_11,Oligodendrocyte,1.0,Oligo_44,0.57,Oligo_44_3004,0.81
AAACCCAAGGTCCCGT_11,Oligodendrocyte,1.0,Oligo_48,0.96,Oligo_48_3019,0.31
AAACCCAAGGTTAAAC_11,Oligodendrocyte,1.0,Oligo_48,0.43,Oligo_48_3018,0.89
AAACCCAAGTCGCGAA_11,Oligodendrocyte,1.0,Oligo_45,0.84,Oligo_45_3020,0.94
...,...,...,...,...,...,...
TTTGGTTAGCATCCCG_22,Oligodendrocyte precursor,1.0,OPC_34,0.51,OPC_34_3039,0.49
TTTGGTTAGCCGAATG_22,Medium spiny neuron,1.0,MSN_210,0.70,MSN_210_2087,0.74
TTTGGTTCAACCACAT_22,Oligodendrocyte,1.0,Oligo_47,0.60,Oligo_47_3023,0.97


In [10]:
#Read in cdata cell assignments, skipping metadata rows and removing machine readable cluster assignments
cdata_cells = (pd.read_csv(resources_loc / 'map_cdata_10xWholeHumanBrain(CCN202210140)_HierarchicalMapping_UTC_1793474000789.csv', 
                           skiprows=4))
cdata_cells = (cdata_cells.drop(columns=['supercluster_label', 'cluster_label', 'subcluster_label', 'subcluster_alias'])
                            .set_index('cell_id'))

#Convert categorical data to be categorical
cdata_cells = cdata_cells.astype({'supercluster_name': 'category', 'cluster_name': 'category', 'subcluster_name': 'category'})
cdata_cells

,supercluster_name,supercluster_bootstrapping_probability,cluster_name,cluster_bootstrapping_probability,subcluster_name,subcluster_bootstrapping_probability
cell_id,,,,,,
AAACCCAAGTGCAGGT_1,Microglia,1.0,Mgl_10,0.59,Mgl_10_3281,1.00
AAACCCACACTAGGCC_1,Microglia,1.0,Mgl_11,0.56,Mgl_11_3286,0.81
AAACCCACACTGCGTG_1,Oligodendrocyte,1.0,Oligo_48,0.67,Oligo_48_3017,0.81
AAACCCACAGCGAACA_1,Oligodendrocyte,1.0,Oligo_46,0.74,Oligo_46_3024,0.98
AAACCCACAGCGGTTC_1,Oligodendrocyte,1.0,Oligo_47,0.83,Oligo_47_3023,0.98
...,...,...,...,...,...,...
TTTGTTGCACCTCTAC_10,Microglia,1.0,Mgl_8,0.73,Mgl_8_3283,0.77
TTTGTTGGTGTCCAAT_10,Oligodendrocyte precursor,1.0,OPC_34,0.79,OPC_34_3043,0.74
TTTGTTGGTTACCCAA_10,Oligodendrocyte,1.0,Oligo_45,0.79,Oligo_45_3022,0.98


In [11]:
#Add cell type assignments to pdata.obs and cdata.obs
pdata.obs = pdata.obs.merge(pdata_cells, left_index=True, right_index=True, how='left')
cdata.obs = cdata.obs.merge(cdata_cells, left_index=True, right_index=True, how='left')

In [12]:
pdata.obs.head()

,orig_ident,scds_hybrid_score,scds_keep,miqc_probability,miqc_keep,dropletqc_nucfrac,dropletqc_keep,x,id,region,...,total_counts_mt,pct_counts_mt,total_counts_ribo,pct_counts_ribo,supercluster_name,supercluster_bootstrapping_probability,cluster_name,cluster_bootstrapping_probability,subcluster_name,subcluster_bootstrapping_probability
AAACCCAAGGATACCG_11,LR_RM_P1034,0.166984,keep,0.014907,keep,0.745124,cell,22.0,P-1034,Putamen,...,3.0,0.088443,24.0,0.707547,Microglia,1.0,Mgl_11,0.59,Mgl_11_3285,0.38
AAACCCAAGGGACCAT_11,LR_RM_P1034,0.397980,keep,0.011334,keep,0.779680,cell,22.0,P-1034,Putamen,...,12.0,0.117348,23.0,0.224917,Oligodendrocyte,1.0,Oligo_44,0.57,Oligo_44_3004,0.81
AAACCCAAGGTCCCGT_11,LR_RM_P1034,0.482892,keep,0.011994,keep,0.778375,cell,22.0,P-1034,Putamen,...,10.0,0.098756,14.0,0.138258,Oligodendrocyte,1.0,Oligo_48,0.96,Oligo_48_3019,0.31
AAACCCAAGGTTAAAC_11,LR_RM_P1034,0.378555,keep,0.013733,keep,0.744950,cell,22.0,P-1034,Putamen,...,6.0,0.062630,27.0,0.281837,Oligodendrocyte,1.0,Oligo_48,0.43,Oligo_48_3018,0.89
AAACCCAAGTCGCGAA_11,LR_RM_P1034,0.321565,keep,0.013848,keep,0.784599,cell,22.0,P-1034,Putamen,...,21.0,0.334928,17.0,0.271132,Oligodendrocyte,1.0,Oligo_45,0.84,Oligo_45_3020,0.94
